# HLO: Reconstructing the 500-Scenario Result Workbooks


In [ ]:
from __future__ import annotations

import base64
import hashlib
import io
import json
import platform
import zlib
from pathlib import Path

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if isinstance(value, pd.DataFrame) else value)
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment

METRICS = ("top1", "hit3", "hit5", "mrr", "map")
KEYS = ("scenario_id", "scenario_type", "source_identifier", "query",
        "expected_parent", "expected_concept_code")
ATOL = 1e-12
OUTPUT = Path.cwd() / "reproduced_results"
OUTPUT.mkdir(parents=True, exist_ok=True)


## 1. Recorded Inputs


In [ ]:
RECORDED_INPUTS = {'before': {'source_name': 'HLO_500_before_overall_Details.xlsx', 'source_sha256': '7734b2c52203cb1b2484ce6ce9eb84635785b9047f2022951c364e2ebb6db90e', 'csv_sha256': 'd5b0b83826ea666e2295b7cb3a9313982d4b5500f10a3029187ab035d13a51da', 'data': 'eJy1Xd1y3MZyvs9ToJSqk5OqkY0BMPjJHS3JXqVMWyEdO3cucHfIhYUF1gBWMs/T5C7Rc+jF0t3YP3KxwA67I5VG1EriNAYzX/9+Pe3cVnlT1L8XC9Xuvu4e11a19aaZW/jcVl1xX9hG/bmxzaOyf63tvLOL39d5A391+PO8ruZ23cHvC6u6eq1/75oiL/XR14FaFl24+/zwNX1ujj43u89XTbP7eP8lfJqv95/uvgz+6YPva9VL9fu6zOd2hfItu279b99+O19uXjf1xlbf3Dffzot2Ze+/fVvMu6Kuqrxo7O/vvr9+/89aXZW26dT1++uf3/TPo5Q+/vmNf/wL5wy4c/rqu6Je2UUxz0vvfXVfN6u8K+btcyn8wy+Y3f9m/ztKEbKlwNVb5F3uLYoWvsnj0+n9o0XwvzHbqU0aHv+IdmsSCUgTqnm9Wm8627zOYRcuvIVti4fKaVUMXw4dqM918/G+rD87bYtYYOpYXT3Af3z9Xd7C81/DwSqL6sFJjIQvRgCnqqnv8ruiLLpHr1vaunl0EiIVECJUZXHX5M2jl7dtPS9y/Dft803q7zeq/00Av8JnP/Dj+MmPBD6K9NPPUOZMQGaj7ur6ozevyxIw8uTVOZ0pgAm+QPF+EXcy1RWcq0+2rNf0HR3eqmaDLZzxYC9Qa5tPxdyeoN6oCHzs9WFPFBWAzE6Oss6fb6sJIQSg12hV0YYGDbATpL73SCkUlfX+/p/f3H7zr0PQd5Dq/A5CIQUQ2QRqXsLhA2tgTsKe7ubjw9fLcnLatosmgMwGFn75WNYAkSdw5LI0AkhtYlRWq021XRnvLm8asJlO8MnlwAtAt0lUmVcPm/zBnhXJ38Om37+x3e/P3hyKJADkMexEkIK2+pM1c1HsWgCdYS/ezpd1mTflo/dmTJKx0x8IwHKcHUnyYXNXFu1yQM+PLUggAMaJUUubl90SIAgAMZ87KflAAIuTBLB4a//CIWqt/Qjr4N3ZZf6pqBsncQRQOcmeiAN2cWtXRcXFvkAAi9PguWjgBN1tzku2O99HWwdFEYDhNFLvV+uS/nMvze28sDA9A2gCAUzOwDks2qNVcto+AuCbBYoO89y77W0b76qq6k3Vf7sB1TDyngSANwtVZR/qDi3nU0v0sINRMfUv50QxDb8rASzOIpU3d0XXDMo2tjKhAP5mRqGldWbuixV2yMdgEEb9YuEs5c1HeyrO+J4N+QgMPxU5V6VX2Q79XlcZ+LCrwXV+i3GIX4t2k5fFP5wPb8gHWA0+az4Hd6T1uto7CyL+0cnpwWzA4fSHPcyQj70avVSCmK3VWeROsBLyYVaDTYhObmvL09jEGbW4m52PsRrML5z9rqgWl84+rJRDPsJqsDkwWgqHByzcoqaAXuutJw06fbwwwyAb8kFWgxL5eW0r76rf19OG5gVyRXz41aBAFps1bWLrHW3nKZ/u5Kjpc0ctEkBmUDSNBQXVx0zAOT8r6fiZiwRQGpCeIrWHIA7D/40EIBuwbFm36wL3fmPndbNwCqFEAoANJ+SuaMB5mdum60MUp8EkxhYSQGvYhtWmadGlObNIYy5eJADWsPm2CDUiwAsXSADNYSsuQOV7E5t7dCsJ4Ljx1bypPy8oBeZoAEUCQG0C9T3YG067wwgAsTHq2nY5rr7LIxsBdDWJqvKmcTb1jACaYkQq7zrbVHQqHqqijz48tp1djfkDZw7H0McoqgDOJmEP/GBqfcwfTjzsMb1jBDA2ifrp74vSLVBvBNAzSdU7DNSvm6K1HnkIv8H3XNab1jrJIoCjqf8k/GL/2hkGTkdWADHTWF2/e/vj1c2t08wCIJnF6hX4zA9V3Rat8g652hb+3NnFK6d3IoCaWapeNfmiqB+afL0s5l6xwrB30e8Z28emFEZ6e0GvhgV18hpiPuoG4PK/6pYWhH68XLrhMx7zkTjwI3ivzebBu1im8Vcb8xE68I16tW7qrgabwHMTzj/++c2TXygcH5MD7aulLal4o6tX9U4ysfhazAfuQIfqY1V/Lu2CEkHtc/S+SK89N/p2L5iP7UEQKbtp6rXNK29Vg9m3KfPGuyvqsn549Mr8rgbToG4evWozL23dFQvrtfbPDYaZyVrEZ3LalXwlEIA//Wo3NyAgblBbVIP77zhCZHa/n7piMV8jBGCAHku1zwy4Hlu+igiMT6l1wGOY3csXn3KYenEcTfNWeQUgjd99a2+dTU8OHl++2ghMuEvddo+7oKeTbZMI6ACwSY5XBQMNhf0EiDKyKAcVtUWU5ypqcIslAioCzIc3YIES5L218C8p6bNZr2u30rlEQDWAenko6zuMuYEhgn+P7vXgsk0II6AKAKq3GdQnacwXCCOA+fCm3+RNZ71fMSwzkI0bAfzBj07wfzAOkAioA9gYJHrrAc5/aOp72+Ieq2xJn9zmVff1C8NsSwTQH/bLrmazbha28eCbYBmTO4olAqgPq34EpZO77//x9QvoDXg99f09JkzzDZhUztUhiYBaSFCN7rCt7bFtv5ZHautEqToEz1MBzZFp9V+vb/JHLuKkAnohi9T1puyK17NdLPgCocZeZCqgIDKjwK7sE8teU9ery07H+eOaCiiKLFG3MPPX/131GPdg220pZm+d4xcL+E9fv8DXjS1JWIC/k7CLYMVrytc5IZaQ7w5NXrglAVK+4gj94Gj+Ki8fMTbh4ECnfMUQogO9wlJttL6fIcggWrws3p7ytUbox2q+M+TWDWqz3vzcnJoLL/cPU75GCP1EzXrD6v3R4X3313yZV6dh0MOh6I9EMHwGhhCZrzlCPxsyAjs7X1bkzrqciYyvIULt7w2V4kAu2QbsHLMbGV9LhODr72pS5YLrGV9ThIE5FMuC9XlWU12g2TO+jgiDWH1uiuFarV1M4XwR3VZZZQKgHvqqs391HuiqNn84I8+FEY5MAOSpxqi0fZx3sPL7nD2RCcB7mKjG5sxqk0wAvDFpvqwrO0QVGz3DAniM2fD2EWuWYR02RXuOrnas2kZKEDIB2I0S1Vqwy09FcdmgWIDHFyVVVQ3/cw5226ZxiUtqPoPRD02o7LquHp0cAC3AYwxNpIoWZ3ZRclqAuxgao267vFrkzaL4h114P23rLH6xzaoY0rujdqAWIDCGYABRtLqYe/m8WJzGq91ejwBwgufyg62+y6uPTjMLgCbYVZiwbe2pVh2dWgAmQRW/2VeSuDgDWoC2GALQr9Alvs/nXd14iwK+CzoEOVEpweTeDKSNJ8Tig2W0K0zvjWqXVyJAQ4wA5PbWJ1gV3bJe4Akt3E6EAP8wglN1nc+XSLH70eYNxY4/F2C654Ac8Bf2hHMyYl5oATpiFCKlvVtastJ7itSwmT4qCB9RozBQr6rNyjYkxt6Z5iX6tQAPMQqNWtV3RQlW+nq9K8E8c7Ivsca0ABkxCjOFFHFwSMFz8P6+S0szOJtagJgYgS3Q1vfdZ/g+XlfXJYOQqAUIiZFJ1NufrvabSK7+UAtwE6PY3/ccAHU934D7dcZsuGxfCSB13LdBaGriMj1g4bFjfZ4WoChGcdRzMt5V8+Zx7VilpwXYiRGYU1R7NvxeJuYXAOZky0q5dY6tawEuYpSkh62ZN6CyOrBoHJ0LAeJhlB4fkX4pPMxUb86wY3ZK6rzTtdVZAlTEKA1Phaubh7w6QyIa06ECFMQojdRZpBt9UQJImyYKlXb98HpRPFAyZl5Xn0AQ17MrgKtpql4dFsLbCuRS2qYFWIYRMULnTQ1oigyY2u2lCDANIwrHlStvXZS1S6mEFuAWGjT6r+3CaVo+cBqY+Lefb66xEul5Yc1pZOZE35+jz2kBpqEBV2qPF0/7N7ygEEgL0A4NuBB3cGaHoqwOlpoAtdCApf7Z3nkBzOOyBnzUNGCOF1XX5NVAtGAMsQX4hAZs3O2rf7YhduWnbjuCj50G9NCrvpJ5C1u4NXg1uVqATWhAR/Vi2WqZbzntLjIIMAcNQPJOuR0qv50sQwFmoAFs3r6hg5fMKkvWAhxBg32NSChyH/oCKqeVEcBXACRaGdqyvywba1+/PUTAzpbODB1uAYqgAVgcEecQ2vBeFNoQYAgagM/w9dueueA0twDsAli9wgZ7VAtwQBpevEeAFWgwtFJvsHfRRE3HRU65AEHQGNCJNamFDwV5YU5aQYAiaGJNOVli3ouVamgBEqGJA5VvAIs/UfeGwSqgsXMuQCY0yOqo21Oni2NuChANTWzUp2Jha7CE51sCLs/OE6AUmiQ4lIeU5abthkiY476ZALXQJDEWsBYfC6fme1qAWGiStM/RUA+YsTpZM7xjBgIaz/8VSioA0eDTvyIPtp3XSH6i3TTgRl++gQTgOUXi/V05bH055d4FaIkGPOw/QFmgFitW63zeeX3qzWlTCQA0lZqs1svyxIkZN/0EGIUmSxV4LRj7kSOOaAFuYYxUxzc7yajnnm2w2ZCTSSFAKIx9jKZWLZh+m/a4/ynGyO5tg2l6t7fGB+LYzzBINiWUVxXLAV9iROP3L3mgen/oJfORPNZaWXCWQaOIlfdpAY5hrKO+Gy/AQlO49V/TAnTBWOOeqz8OmGwT+52P0XEQqtb+hbQFsMn+f7ggAypPgEMYg4v/avsXeQMq70NdPj6U9fNKao7VK8AijGGOg5hYyOC9tdVpjHv8TQswCWN0uZ8Kct1bd07etgBLMA4ThdXGyOQdqOwYnV0A4cFpfPX90fTerF7Zep13y2LuthQC0B75J8L0BBmnFIgAKTCOtFra5u4EBDiZdwHGXxzF6tWHZd6scrBsa4t97F7+xgTAOkrUAv6MluPJ5p14SwJwHaXoDG3aeVOshxTVxfa9ABsvRkL59ZE0ZwBlYlUEMNb46kNjh5X3IQQ0zIA68/EzyxcEFeDkxSZQP9nP7TpfO+Z9BUh4McYEDt0HXc6NANsuNpF6qD/ZpiLD9ahFnpM5LUCxi41Rd02dL1rwkd2WQQBmwdnHBFZbDj37xPQCaApO5Gd7N8/bc1buRZFSATJcDA5Fky9ypzJKAWYbPkRnS7scrG8dnVwAMUEFvLFl6X1AeobT+RdAydSnJ/9UDCV8Xq7kBZhoMRxrFO0+n7fFqnjOMRxfGgHmWZyFff3HfVGuJgKvbh6EAPsszsyz3vdD16E8raga0GiDQvY7W4CTFmfpk1jf8R0lTi+TD7GJH6pFPd/g/3UOWAuw0BI/U4dORU+4HU4rwcfYRAdqnjd39fJxgZ129sI4ycHH3ESn6v2bt6+1dpqYj7dJEKi3t9ev378XhDs+EidBuBXrRK6RBQkECGhJgMeUKBXUnqTqBukVo2ckEGCiJbDCP1399Pbq9Xtswre/4WiUlHWpbxMIENaS0KgdU+znTTevsR3Dmyd3HJ0XdtSOCARIbUmYqE2Fdy8uvF2Kbk9QHupO6sIuCAQobglGEzbgqXoPFgmPWDfwB/jMYnHfQID1lkRPL5UZur5u4iQIYDT4rgMNDLa15B4dD+pS0t+x54QXAshtkm1tPV7c0hXtSSHX6E4XQHBwWN72t05+OOkhNvFyBIA6xug43iHqOXYwCwTYcEmcYhMOdJfP05mcTrYAMS5J9KEr3us7un7ytG3OOZd+l+o9f8FPIECVS5JtF3zq8Np3Aum/PjpJXl4tvP4ij2L4bB25xFuhh51iFFoA08E33HYHGe3XxdCLAhS7JEm3wN7Y/h7gSREnzokAjoNje+hhetI30kkaAUDHME3Pod2+Ttxoi6K1cFa8dtN8skVZYl3sS+QTgPQ0VvDuLFKFLnh3Y/jOJ9YBIN3Ud88JD4xqrIDPrIM1vl3lTefNHfUtn00HC9pPvQSbyeVN8Gl08B1ua7re6Sd3gkPAp9EFEaDT/PEO8LopTh9+dHI2+AbwyjdrjIxhJ2GYtmvq0ruCc0vEvqv5n5uibyHqJBcbboNMdfldaZ0sDz5dLgzUiCP24jhTwCfOwQwdVWc/bZfgtDxsCAXXcBfCf1495hJFD/j8uTBWm9fDnRpG0YKNkZF6k7dz6nXk3XaPpfVul3agnu3FgdOAz6mLQvWbvfN+7mMcj96Pg63FxjcLn1wXGaNVWRde7P2RV5+wVF1nSer9HT/bq98/N9azHV7wbJvu65f2pDnAqJBs8I1ihV94v/XX+nk/wP9ZO4nAhmCDFz/XeAmZY6A04JPrEvXkkl/vl5d05Av41Lo0yMKeSv4BO8bT33s/U7/bwTTZhV4fm3kXqKsS9qV3D8vzsHE7QFy0DRLwpKl3hn1yXJyE4AItAOZRpZlYDItNtgsxONKz3H61TQsrA0j8tmjnLqvDptuFPvzA3ncr2LRYktpuyqHMy7hSYvPtSIxU/bnJq26z2rOTXEJGbHYdygA4tm+s8gneDvl6coqRzb5DGcHaBwtmUa9eb68yXcHLe761GVmSgE3KIykj9V0J/zh/HNzUDhDIZuCROGZHIGgBiryB1qlO+52LyiRSpnZ8Uu+a2mW1FG4A/fHQ5CuS87uiPpeJdYhwsel6KC5YKv++pWDU93vB4E/vz63hxHnlAjsJlajvrt/sateORaGl3F9jcZ07XnkcsOl8JF6qdqL9DZdsl7t7Vz3AG7UN2mt/27KKHeVjk/pQPsDt4yWDr7a940CqbUThTf68G80p3WTAk7yoOXnApgXSQwT7c9QeHWg8PtssIMPPYxMEScIYNJttHvd5Pqf4DJsOSCIk6vY/fnSaVkILwC6lmpIztzmNIwSbBUgSBPuCktdY9VvhpUUvuFsqYDP9SBrq1WkbvGLnma9wYx82pWvNS8Cm+pFUiao2XQOQ7ojhbGofzX7UruVcC/HjuNV4x6mATfNDmeJU3WzunFxINsUP5020WhQteNJt78327Pw+4bC1Sxlgxib7kYixyucNtgNmZLPYbD+SBOthHpxjDmxWH86d+uqXpv4jr7xl3bgBCZuNR/OH6gNsEcyUvce7nw+VLaCwXZpBBWwSHokTqXe7m/72FRHtpqGd6yyRBNKmiZpRrvPD0lZ197i2+2iikywS+Io36NSLofTveWhls+xo4lh9d3X7/o1DlX7A5s3BxNhvvS8Os513RV6qy6KzGXMkglZX7eNqhZfuzb1dqOV2c4eMlzvQNT+6lbcGbPYcSRWoet33AL5HKdzWRQA1sS/7/tXc9uXIaIZg0wEXOn3AZtCRMIma27Kkattthy4nGQSgFDtvv+ubXW1Thjc53lGHztAPt9feu091uRk2yybDQGdih2xqHYqtfRX94BK0YBPoaFatzA8Mxc/m0ZEQRl31LayrQdf/qm3ReSV1eFUhOdq7fVzhTZKbydrKy8NkbBYePkoQUOr+65fu6xfve9CXX/8nx4vQF9b75euX8uuX1dcveN3U+fjP4KtmY3hwCE73HX8IM+7zoWjoxfYnm4UX+eod9vxv6tXjmcYkDnuRzcuLAM3f/vrWo85ILrjF5uRFgOI4M7WKkdvSbIZeFO5eUAVK9wPoNyoNv6GuSC5WCJuuF4FFuobVWRWjhRBaOV7cHbCZfMZX37sn49i0PWNOapWHnP1JnXYmtcGm94H/PAMrvfF+wVuprvPm42Z9Nu/vEPxmU//iJKKGz6uCui16O8IDgXT5L8/rABZgO+xrAQTXlwvp4FA85Ws8Y0TU95ioscO3yJ/3U9gMQvA3nso1odFdDg2bXgg2/1HS4yDZ1n6+2dZiOi0YF/vB8P5oH+/qE6/aSfmxeYNJHGPgrmlsu66rRX5c8Py8JOYFvP6BMrSBw82mHIKlNLDfnPYYVx+kePFJVWBPinOpijM7iasSUmzdtfj6ZQ7uVw8CX//bK3PvzU/vnVIFbJphivcpuZcdsFmGGUbRnGJTbAZhho2FwXKlm6eHFvrF1lvI5hjChtqbbG8GSzDGFucG3BnV2G1Ps9/LugYN7lrhDj/UDPt196NDoQBMHwhMD1ZSAgjbj6epzks764E0IV8a8PFnUYiLgePE4kcC82l47lirfnSg/tzgwrGnB2d4loKP148TjxsLzJfAg/o4H45uj5sITJ+qGRJy+3EM6m6wOIo/XwaHKsH5cHw50oA0GV+aEKTBNEY/OsQRbvBYsKcHi2LmJ0b14/he0wK4lsBRTjICFhidcE0L4Bq88JlPsIrjmF0B8wkgVwJHOdGEJHpqb2sB5AKwnoURzocjZ29rASADk24Wxbi5cJxYbQEgA+dwhh0l+9HBPb7B3cCeHnzAWQh+dj9OPK4AkIG9OcPaqn6cmE8AqsDOnEUBbi4cx7Ei4GMT3uQ5SwLcPjiOH56Aj02YhZkhsaQfxwJzN6ij2fNpPB40X0Tzjb2/gA9GeOXiLDU4H44T748PRnhD4Ayb7vajE9YHfPTBO9dmxsfpcZzYPnz0QWLXLDE4H45O6BPw0Qev45lhr4N+nHhcPvrgFTczn4xyf9IoD/jog3ekzDBr248OQZ4bPxQAIwOWAyY6+nH8cUMBMIItNAvp8ITx1O4NBcAI486IgP048XwCYBSjj2VS1Y/j4BcKgBEaumGIygTHifUUQB8wQGCnkM86vZ4C6BOjFxWS8gqnwD0UgBs05FNjVD9OrKcA3KAlHYWh6scjF3XEQd1NL4A+KWxXvJGkH8eXNxKAm0yj2gpUP07MJwA3GSyvIVvSbG3Ji9E1EkAfNGVDckPDSTc04qMPtteaxfS4OE7Mx0cf7Fozi8HI6MeJ+fjogx1oZj6hgT+JBhEffbDpCKykVv04juYRH32wicgs1LieOE48Hx99sCHBTJPrg+PT43GhT72Thg9G6J3MUGH3o5OlafjYhKXl4NCT3xJPbS7DxyYsKQcUpICdP+V3Gj4YYUHMDOsb+nFiPgEwSsFxiDIyFTLHWL8RwKY0wgeNVD9OPK4ANmGUJkwS1Y+cs2QEkAuDNpoyHThOPL0AcmXwsmOKUZGB4hJsNwJAlqEZTNOH8ZRZagSgCmNGeG1VPzpBVcyHqhDzWIbCzTiO66WYD1WhxvgrhVhCCrGMWcUxH6pCDLGkMdoxOB720i5PFwxbxUOrzQcydKxmCQFZkk1trpiPXCH4TbOEjORk0kiO+cgVYq7GRJSUjUbjyzgfH5tCDLFEFFDG0UkxxHyoCsFUmQXkcgWhY2oq5kNVCGg7MxT+NU/Cv5dABx+5wgQDMBSgwJGTSkkEgCzFPB0Biz8JLIkAkJE/SGlJHMePViIAZBmmQRPKZSQOq/38Y5SGD2RRnwlIVD9OPD0fyCIfnj7OyD2E0cn7Tvi4FpHWSgLVjxOPy8e1KMD5tK/68RDJ2dlAJy/7TKtEkIYPc5h7maFR2o/+0fL3dS5maPEHQTfhox4Gt2ZxEKl+HLdYEj7MRahjsPdrPzptvZSPa1GECXkTqH4c33opH9eiKEYvi1Rqlh5vPDXBqIbp+TAXYVQfu33148TjCgAZuurYw7ofOXs7FYC5mBLWserHcSWWCuBaQkFbitoYxzquVADmyIIgcxFHDsylAjAHimIWk5uP43h2OxUAsgR1KJ1sHE+f/uxVyTdYx8GeHmtPDBmvJpzy/DIBIEO/PqB8SGCmgkaZAJChY50G5PkFjr5BJgBkaK/hTX796KQ2Mj6uGU3pEXq7fjiBJBkfuUxAakqrfuT42Rkf1wxFow3Ff40/tdf4QGZCzGVmlPvOAseXzUcug34o9vfsx3HkyvjIhd1OZz4ViOLo5PdmfOQyBi3AMFD9OPp2scEAe74Y1RTVRIZ6wtnRAqXtBhVDTN4Gjs+QOR5Y13Ofozx8KDMJ2kgUQwviiUoELVDNHvtYSRJRJjua0ExaoJo9xlynT9kiHBnxdi1Q3B77WCXlZ6ofDzi6Q9KLmQVaoPQ9xvgwdm3vRxctqgVK32OMM4SkxkJ/IsqiBUrfY0wLR1RUhCNrK/CBLsawQxSQWgmmgE6g1j3WVBDoq36cmI8PdDEmqZGg0Y8T8/GBLMZkQEQFiJGZcC61QHF7HGC9M9mAOI4aYVqguD1GsyclfyqN3PKkWqCaHQs3Zj7pCRwnllcAm0D5z/BumX6cmE8AjELCQkrr64wFDgLF7THmdkKKS4XaLfmgBWrd4yjAQshU9eP43haodY8x2RHG9LhxeOQr7wI1lzZmuNEClfBxTOVRmerHcRNFoBI+ptgEIVdshtTwhdsPpRHAtQS3HsUIcWRJI4B6WJ4REwzEkzAgUDYfg7s88ylO5KeR28ETqKLHY0D5+jAL1bG12h+Dk6UfPBVDb4IPkImPCoAMdxwn3gQfApExBpouUP04gUF8zEtAR88MJUJwHJ9PoKQ+QftIk8LBcXw9BUrqkwjLEyi1kgSHCOh0EAen54NcgmEMDP/048Ty8mEswTCGpmi7DqfMT4EK+8Qg15GSmMk2iXleaQhU2CeGAtpUc5pO1JxqgQp7pCnMDKXgzRR9SgtU2CexwdIDwkJz4ODvns8hfaAF6u8TVIqG/HYcJ55eAIywcamm2hedThlg/IJ7jbQQWupwCtn51fZwTkKqawmDKYXOr61HJw2vsuhHJ3XOL7THRH9E1YbRVLWh5pfZo3+IPeX6cWLLsAEoxOIsWtgomkjUa36JfYg7lOhEoT/k+15uk/LL70FZzCKfaN3+5AZmQw8VKFDMKoinohr84npkGcakpONs6tn4tfRokfgU6vcnQ/38SvoYw2yzhDIbSaKfeZkD6uTCSy1QODZOIX2NyrCD5LnhP2T0DwUfBiM//AL9NKN+MFScliRTkR92RT4CtiFf2ExG8djl+GAHz1JNLoY+ycU8AzF2tX1gEDlSqqtLp+rqNLvYPqDMVkQ5eRwnno4LVNhFxydvxteZ4iA0u+4eO+rExI6Kp9hRml1lT3eWIaUjorL3aMo4ZJfZ0wVlWEFIrmqYTk7IxSaaMMES+1j148SEXMiha7HgCVMySdNJk5RdS08T0v4l91Sb41KpkUKprTHFrq2n+bGgJaEVTo6z/NPFWppda0+XVIWIuMS4napE1OzqepoQ25uQP5lM+pPsenqaEPt/kLWRTlob7Ap6utKGAliR6sfxCdlF8jQhWm+kX3DkuMzsGnoSB7c0GcrppKHMLqKn+3qQlE/An04CP7tOniYkxg9ZRtmUZcQulKf7TJDyo2lC7Vb2ptmV8jQ/vlLyaNN4osuKZpfK090l1IuQHngy68iuhqcJ0Ueg+H+SRm56gF3/TvNjGpIyb9Fk5o1dAE8TkuIhlEqmUIpd8k43baAVTM0H0pCXX2JXxJM4yN0n04acqKMFnyqJ1+ySeLrGIkYeO0X6nnAYL9hw7BJ5unsCK3cJUnDkpF7ZNfIY5dEUZNWTQVZ2gTzm+2Oi5uJ4tAknqzo1uzqeul5SSSeOT4ost8becGQS5+aCHOY3qXAaR6eAKLssPsKjb8hRwtFtzblYF/XKiwKkaTplALJr4tH4Q1pfP05MxsUxzD3Fma/6cWIyLmhRay5qBhJO9UHU7GJ3LAwJSDsFk9qJXemOdRgJVXQlEQ8K2WXuVCVM3ftwHI/dsGvcqXMnZYZxHLc52BXtRK2ltL6JJk8hF3HQmgqplzSOE8vIRRjs4pRQjzUcRycL2JXrxBimLjR+PF6t8H8/quug', 'recorded_summary': {'top1': 62.3, 'hit3': 75.8, 'hit5': 83.1, 'mrr': 69.39999999999999, 'map': 70.89999999999999}}, 'after': {'source_name': 'HLO_500_after_overall_Details.xlsx', 'source_sha256': 'a7d0225e61eba3f3e1fed1682b1ba7a0ca2926cd4784bce350c6aa98992b736e', 'csv_sha256': '54efb6e5ceb73b3e138caf5c757db2138d10c6218ca6674ff53b9b971da64070', 'data': 'eJytfd1y20iS7v15CoQ3Ys5sRNmDwk8BOHdq2930Rqvba/XOnLsOiCyJGIMAGwDt1jzN3p3j5/CLbWaClCgRRKmUaYfLMm0rE4WqL38qv6x+aZuyq9rfq5XqD18Pd1ur+nbXLS18bpuhuqlsp/7Y2e5O2T+3djnY1e/bsoO/evjzsm2WdjvA7yurhnarfx+6qqz10deRWldDfPj84Wv6PD36PD18vum6w8f3X8Kn5fb+08OX0f/6GIZajVr9vq3Lpd2gfuth2P6fv/1tud697tqdbd7cdH9bVv3G3vztXbUcqrZpyqqzv7//8fLDv2l1UdtuUJcfLn99Oz6PUvr455vw+BfKjLgyQ/VD1W7sqlqWdfChuWm7TTlUy95Li5itBc7eqhzKYFX18E3unooP979G8eGb6PC7efQjQ20SAW1itWw3291gu9clrMJVsLJ9ddt4zUrK10NH6mvbfb6p269eoo2AaKMubuE/vv6h7OH5L2Fj1VVz66VGxlcjgl3VtdfldVVXw10wrG3bnayOWSVyASViVVfXXdndBWXft8uqxH/zZI+E+5+4UHGBhpMLFD9GrQoBrVJ13bafg2Vb14CCni8Hdj5fA3M/Lwcl2ga2yhdbt1v6jj4KsfETtm10r1Bvuy/V0p4AmQeSaD66hnGsqgZg5KBW3ZZPF45jWgTANdWqoSULGH9QpL0JCParxgZ//a83V2/+3UsrAZBNI7WsYT+BgV+Sdqf76aBA+CY9/Mrj4x/JxEeongD2pjDx67u6BRDkmCMtgMWpQXO02TX7iQquy64Dr8hvIQmgcZqpumxud+WtPatD+EgHfGv73ydekwA2G1iJoAWt7UeT5DU3AnAM2H61XLd12dV3wduXahIJwLIpjjT5uLuuq349YR2evqj4yY8zH6OSAlCdpWpty3pYAyIBPpZLL6seCUBzlgE07x1e2FO9tZ9hloJruy6/VG3npY4ASGfFI3XAEe7tpmomV5AH/kQCSJ1HT1WDqOd6N4XZD7qF+xkKH2ZJAJXzRH3YbGv6z6M2V8vKgvhTRR6Mx5mVnejH03VQUwCxCwgOq/5o0rxWkwBSF5Ginb8MrkZHKLhomnbXjN/Oy3BEAihdxKqxt+2AnvM0Ej13OQsgdZGosruuhm5CmWctnKmPQbdYALuLVKFbNjlNcy8p5kMySFe/WdhaZffZesvnAzL8VBRc1UFjB4x7Z5bKI2zZB1UxH4U1hM7vMA/x96rflXX1L+/NG/PxVkPMWi4hdumDoQ1eCCIxH2w1RqkEInuvsyonY94HvJ/YGuEk1KKCfJjV4EBiCNzb+jQ3MW+LYj7GavDVUPp11ax8dwsfUjX4HJgehd0C/m/VUgavD7az7t4zQTbmg6wGq/Hr1jbBxbiQz7uhc/OU8BFVg8VY7ba0iG1wtJy99BBAV7AOnQWbMyZJIPx+qS4CSAtoTdnWh6yNlwYCOAvwtG77bYXrt7PLtlt5+SSJAMrCKr+uOghAlrYbxiTE03TRPIYkAhALK6vZdT0GHmemwWPfJgKQCstrDyuT+nAc60QAc2HtrcAwB7PL1/HaBMA3DdWya7+u6KDK4USeS4Ac1BEA2zRSP4KT4LWFUgFsTVN1aYcSXwYjDZQKoGuaqabsusk84stXbCqAtZigKofBdg3tqNumGtMNd/1gN14vTAB2s3gEfnCXPpe31ku8AORmySj+pqpPofZh1551I6c+RtUEgDjL1XtM1G+7qrcBxQD/gO+5bne93zQJYHAePsq32D8PboOXIgJomxt1+f7dzxefrrwkCwBsYdQrCINvm7avehU8nMb28OfBrl5NI/64yc+ZyIN+Aohb5OpVV66q9rYrt+tqGVQbTIpX4xKyY25KYR541PvinN7PWPUn4DSuesMH8AhC/lfD2sIz3LmVnY++DR/FozCBt97tbgMPneZWouHjdxSm6tW2a4cWvI3AT7nwaF2Ox1fR9BHWhC9n+Ggf6VCtbU2lHUO7aQ96i5lHwzcJkY7V56b9WtsVnSn1L7IL53YI3y5EUaLsrmu3tmyCTQvu5q4uu+C6auv29i6oy+sWnI62uwua3bK27VCtbNDbP3aYkyYvFZ+J4dMbvj2JIBZ/dVAFwBRXs60av53EtyYReLrHatwfI0yi+YvXJN/4RGlIh/YA7aBbUK6+lKDY6jgTF2zKBvAev/vek/PyvQ3fAkVpfDgUHu4OCdMz4euzrGImYE7A+TmeJMx3VPYLwM/kHHGyh5mAtQE/5S04woSP7yz8SzpO2m23rV8VXiZgZcBS3dbtNWbzwOPBv8eUwNlpe0DEdD9HE1aFdBMwIwDz+6PaR+el/ss+E7AX8OLflt1gg79j7sjLNc8EzAG8axLfB4DzH7v2xva4bBpb0ydXZTN8/+allAC+w0s+lHC23cp2AXwTLIGaXj0euYlMAPVh1o/AkruCBNAdZry9ucEj0XIHbpF3AjYTAO8MTeEBcvoRcu7n48i4eHnguQCCF1r939efyjv3izpjXPaTlAvgc5Goy109VK8Xh0zxM1bP7AQJAHWRKnAGx6PjoGvbDVspAYQuMnUFkr///80ITLe235diji41frGC//T9G3zd2ZqUBcyayMLMZU5zPn7HWOh9WPhldZpe9rBtOR/P4zA6Uqcp6ztMN7ysUGrCS8n54B5jTLzB6mt0mZ9Axqzf7MD2nI/tcWjU8uBBbTs0QaPftzs1zLNbgI/qcZipxeikfDjaj+//XK7Lxi/RmfPxPQ6LKZdpsMt1Q4Gjjz4FH9ZjHd57CNUDyWOfVps4v/AoZS/4SB9DkH2oK2Ws6IIP73GUPlS4gp/n5ZwUfCSPI6O+dtUL65LOhMSFAGzHoRrsn0MABqYvbz2LLQoBoKbintqO+dbJguxjiJ45bSoEMDnOVGdL36qPQgBx8eB73UK478nRKgQgFs+7+zusLIYH31W9tw4CwJpkqrfgH5/K9oEKLHXjq5KrpoX/uQR3atf5mBjN5wqGcRoru22bO68ASgswBuM0UVWPkn3MmBZgCcZpqq6GslmV3ar6l10Fv+xLJ36z3abytqxagCoYA8pS5rdaBuWyWp3mfv1ejwBSQkDxk21+KJvPXpIFYBE8Jzw47a2X3dQCPMEYbO/b+2oQP+kCyAjIvsFI9aZcDm0XrCr4Luinl0RaBMd451ulogUYgsmhBHx0jH0mRYAdmADI3fuX4DcM63aFO7Sa5uS9KMDSAqTBBPbcZblcIxHuZ1t2lHj9WoHrXgKuwF/YEyrI849vtACHMImRaT6sLTntI81pymtnzSMfnJM4Uq+a3cZ2pOR9MO08u2cwkbQADzGJU7Vpr6saHP7t9lCReTYv4X2SqwXIiElcKCSBQzALMUnw18PB87/7IIoAEzEBp6Nvb4av8H2CoW1rL6QVYCAmaabe/XJxv6Y4PqAA+zAx4X2jALD8yx2EbhPn6zPvRADlzdisoGuJcXSLpcWnBXnPih+nPgYlBTiJiUlGXsX7ZtndbT1T7lqAcJiAo0a1Z9OvySFfAMWzPbPkyn0q4/l6BLA7yx/WcdmBMRzAk/IMagSYhkl+vJ/GiQrwAHnne0qjBciGSR6fatN2t2XjzwvSAqzCJE/UGeBzyBYA3jxTaNLb29er6pYOY5Zt8wUUmZgHD/9IgFyY5Ll69TAvwV4/n/IWLUArTIjzuexaQGKkvLR+70iAPJhQoq/eBNuqbn0qGLQAezDFYOPSrrzE8mE1BcH/+PXTJVYTPS1/cQjnw2YKQdo9QDzu0DBdkOPQiA+gKcQb17BJTzO0sycwWoAtmIIv/tVeBxF8U5+H5uNiCh521Qxd2UwkHmafmo+LKTi1+3f9ZAUcakb9lgAfDlOwNK/GauQ9EuFa4BXSagFGYApWaFTLNutyT0T30UGADJgCyh7M10P1tpcrKEAFTAFu92/oIUr2qyU+cQXHgHSiDnbC5gowCFPsa0RPQKHHWATls/UEOIQp4CVNI63v39adta/fPWTefEpntACfMAUMnVHnIQ8SnM2DzK47AYQGrI1fvxupCnI5GAHWYQq49wob7FEdwQNouVNHs1MmAO+YeGl32Ohotjzkmd6uAPUwTcG8tmRhPlYUo3kZGAG2YWo0nQUT814uTSbAR0xNpModwPoX6t4wUV80PzkCRMQUeRtt7xuhCRANU5OqL9XKtuAJL/fkXb8Dai3AMEyz6KHSpK53/SDMydQCPMQ0M1iTWn2uTvO8x4o5EogCtMM0y8eTI2r64vWuBFAXYvpXFLL2yxb5TLR8vCBWgGSY5sjDv67POWNzZluAaZhC0PxPQHg0PdVmWy6HYDzFkztcEeAbplS1stmua8+zVQGSYFrkCmIazP1wdosAP9AgXfHtQRVqsmc7bBjktWYFSIEmxHxp04Nzt+uPG55iUuzGdlgA4Pea+MhrwgLTYC6lgqZa+00XH2+N1spCOAxWwUsyH1+NTsYmubC1u2qi8ZmH8ybAxDMa1037ecJ1ckwFH2dNFKve/onkAvCNWBPBR10Dgfir/V+UHZiej219d1u3p4SH2VnhA6sB4H5QBGsTgne28U0fC3DnDEazjxW5HP0oP00EUDbOFJYII9F1olhjVroAtELE9erHI/HBot3YdlsO62p5NqZ/xqoVYMKZJDzRbWSm+L0jAURNtFrb7npiI89KFkBUCPdffVyX3aYEp7G12BNu5hWFxz/fPPqF+ghAKkT5K/gzumwnq9XLQREguxkI8BFf+2VXbZ8amyNHdqISQIDZZpBgfXkk/mUIIgCqEMt/7Oy0xZ2TLcBgMxBK/2K/9tty6xmDC1DWDEbQD934vKQLQCeExrftF9s15PUd9ZcTrLgToKsZCJyvu7Zc9RBv+k2SAHJC4IxHRX09Xef1fMa8FuCfGYjPvtrrZdl7bhQB4ATHvitXpU+jay1AGsP5G2xt195FqQI0MQMg/9bWdfARSRQMb0KAJGbykCbiSzV1eDI3EQKMMAP7GIXflMu+2lTTfL1nIMTEeYAAIcwU8VhLcVPVm0l62rFV97gZQgtwxEyRPmkcP33TyOwbFEDRIn+UNzu+0cMnhybADsvCWK3a5Q7/r3dhlAA5LAsL9dCb5xEDQ87wCfDGMh2pZdldt+u7FfaWuVfVa774EJzpXH14++611l6C+fCbRZF6d3X5+sMHL8F8rM2ieC/4w8sjhEiANZZFuGuJB0HtOprhDCdiZkIiAfpYBkv9l4tf3l28/oAt6+6vBzrLpAqPYHc81jvZOpMU4VMDEQlw0LI4VQfy16+7Ydli44O3j24TeiEpLBLgqWVxpnYNXly4Cg6HYfes4unGn2ddD9RIAKExX7CD0DS4tUhaxFP2f0LM7DUxAkCdPL5zZeo6Nx/yeSTAWssgWJ3oHLAv0w5oe1DHj/EKOo+YPhIgtmVpti9bx2tOhqo/eWUztj0S4LZlEKG8Gy9l/HjSF8snXIoEGG2ZwRw23rgZnDbp8oJyAYJbZnLsfoGh9nni0PyWEqCvZZl+6AP3+pouazztSMN4bQIMtizb95un9qhjR47x66ONFZTNKhivvaimthrDd4sE6G0ZRI77lh6z7ay8Xr8AsGf5Htg7O96a+wydGBMpYAMg7H3o+HnSR9EH3gQYbRkmcUaW7P7t4jJcVb2FnRT0u+6Lreoay1V9DfdJziAS4L9luVHwoi1ydF7wos+3xYj4bDgAs0/t9VPiwWmwPuE3TsTuU4uPbUDgDVxtym4Iln4EsIhPgYP5HUWvwf1iLCI+FQ6+w1VL1yz9wu4aGvGZcVECMLi8uwbD0FWnUzP7VtiwHsGC2G0xBYctfEHs0LV1cAEQQFy9i+Ufu2psx+mlFxvXo0IN5XVtvRx0PuUtjtQLoyY+vQ2AYKCi6cfdE7yUYAMshJWHw4GnFWA+rekiPq0tNmr32rtxQ8QnsiXqbdkvqblRcDXc1Ta4Wlu/qrOIT2ZLYvUPex38OuZH7oKfn9Ee7JluzEFHNpAmaapV3VaBCf5ZNl+wRlwXWR78FT+7N85/7GxgB7x32XbD92+9zz3LEZ8hlxiFXwT/GO/TC36C/7MVnEU2BKd4W3OLV4Ux8rVPP0bN2CCcqUfX+Aa/vaR9X8Sn1uVREY/c8Y/YuZ3+PviVWthOYtTZNzn50QRnaML3YtP0InVRwwYIbmAub3deZp5N04syCPepd4Z9tC8Z+QM2UQ+A+qj+zWc2uAgfY0Jn5MD93XY9TAVg/LuqX/powSbjxSH8wDZ6G1jSWKHa72rvw6OIzcYjNXL1x65sht3mnnDkpQMXo1EHwMD7xipf4O1QyMlYn2x+HSoFkQI4Q6t283p/HekG3pbfamUT60iPRP1Qwz8u7/zXKRd9SX56oAj0gB/BRCdVuQwLm01HChfqwCcNLqnzVk95DbAft125oaf4oWrPHRfPTigXjFE/cGv+Y8+5aG/uNYE/fTg3pfOJdjavjpTK1A+Xbw81c8eq0Nzd3xBxWc5dTHx44zMhPJuDR9rm6qDpX3AGDyeK75tbeKO2Q1/vL3tW8dl2rSOUOEJ8NjEP1QWgPp5Q+GrflQ6U3Gcj3pZe/WYiNluP9Irud0p/tKFxg+zPF+V8ZTbJjxQ2YKxsd3d/oCiIPWz+HymYqav//NnrRUrYCFilVAdz9s6lWQ0krAS83kMRzGusRW7w8qAXaSNhAlLqAmo7vNvmSejwyd7u6jOszPMYy6b6kVaZanZDB4Dv1zs4YjP9SPpRu5bpfuMOJSSA2+Tq0+7ay4lik/hQbqbVquohzu7HaHak1I8nG3vP02c5sOl8pJNR5bLDVsJeoiVwNMM6nNtn5RxmTk3YvD1UJQ/Vb137z7IJ1m33FCq8TDWbq0fqxOojLBE8kvuAV0E/VNSAhfbp7xSxCXykTqLeHy7gu6/T6HcdrVxvjSSgNc/Ugo5cP65t0w53W3ufpzxHfsF3d6ib0tNeIZvih6rhlTnt6sUnvk8/RrUkgLcw6oeLqw9vGQEtm+sHemDb97HczQ7BBYW0PsfNbJYfqaDVRX+32eDVeMvgkIi52l0jaecaDNPPJwGZFwSwCYCkZKTa7dhs+AaV8jLVbBIgaRA/vKmrscAaXRjsUHCS8/SZHQG8xn7lS1vXVGy87/3lNT8CII3twd+PbbT2R5WfSryUDuOqn64ug/df2nrnm0xjswNRMR2q5CcvqQKQjLUH6U9ykRqbCkg6pepi7IbdTCYVLvoeA1+yrBcNkrKDq7sNXv+486oiYjMHUdkooqKA79+G79+CH8G4fv9/JV6EvrLBb9+/1d+/bb5/w9unprDp2RjOphbG0UPqemzxQyBxU3qV0EdsmmESqvd410DXbu78e8hFbKZhAgD97u/vAmp25CWZi80JADNKplYxXpK5yJvEhzlvwHB+BKNEBeufqNGRlyZc/E3AI93C828q78NANmUwDdWP/idobLpgmp7USPvF72ziIMTAC/C0u+A3vIbqsuw+77bPqgpwYBKbVGiyhNoybypqghgcqBSEnfX/floHsAIrfV8L4DWDXPAET/4x1+MJX6K9wdMVO3VljUdym81NBEf/sZoO2+kzh2xqInjXRwcXD5rtPdVP+9JMn2wGm5EIPu1ne3fdToTCc9EMm4KYGYP5tK6z/bZtVuVxOfTTohevl8TFZ3AzJtaMlwpcmM7xTpGmwp4UXi34IjbLMMcmWavv35YQnYz7+vt/B3UZvP3lg1dSnk0zzPFOpOkTfR844cJzgdmsyY2hjzR4/hkJm4xYYPtfcBnpymfPtxKzGYiwvu5dp7dnih3Oo8YniBRUZ/fNxH6v2xassG8NO/xQC+ybPY4e8TuIjwTEgy+TAWiOo0e6F8THfPEQIC+SGJ8ex7nXDfISAXkaHtRoNY4OeSlfHkSSixwCpHH08MVAvBEQn8GDhigexznT+2msP+LKy9UCCbnj6JjeXEBeAdsmQ3k4Op6v4MuLQR4eCYzj/PPB0mbLMwYkZakax/nn0wJglMF2zApCg8L5fALoAy9tERL4hXvwe/b20ALok8HuzDShgXYtVy2APoCwizhBeTg6XqcA+oDjtUgMLh8cHfIE4AaisgV2ZxxHx3wKwA2EV4sYYtZxfHmOE7QRACNwEhdYiTSOXpZcC2ATOIeLJMLFheP85Ed8bMIbNRdZhIsLR4c8PjbhQcMCGR3j6OE3f0IrzBavcS+R+ITEzz4uH5vwgsJFnqI8HOf3bsTHJrx8b4HtbcfRIY+PTXgl2SINUR6OjvnkYxMyqhZZivJwdMjjYxNeXrPAfgXj6JDHRx+8EGYRkieNo5P78EzuKSrHxya8UWSBp5jjOD8ZsQA2peBX4LHAOHIMQyyAXLDeFjFtrfhoaz3LMMQCyIXJYkTPcZzf2bEAchmMsdJcjaPjZQsgFzrJcYyGCEeHPAHkgpcDy4pi1tC1s2MB5DIYVMVk+GKX5YkFkAuDgDxN1Th6pQRiASBDpzyJYzWOjscVwKYclite9jGOXrszEYCqQqMFjNQ4zj9uIgBGBUxvSl5qGj/1mR0hVyIARugkxxTRxvuI1onMUx+jNnyowuZbC0OTgaNj8vlQhS1uFga8mXF8eSUaaMMHMmxmswgJWHD0W/l8XMOGJTDtWo2j30rkwxw2IFnEGt8Fjh6EdhDPhznsT7DQFK7hyHFREj4IYkS1QEdhHOc3QspHPSweXySGoilT+L37lA+CWEMOcEt5Q2eeMuWjHta4LLDcYRwd8gRwLYdgJynIRSlcLkoqgGt5gk+WqHEMj3++efQL5QkgFyaa4ixT48jB0VQAyDANpekMRWcujyUVQK4C3q6hNBv1/vPB7VQAuQr0t0l8bA6Tv//1EEscXrYANmFiCy+mGsf56TV8bIrxSCylJDiOHGQ2fKiKNeaMKQ+Eo+Pp+VAVYx4oN+gS4eiQx4cqDNkWGUEVjuGxIX4zZ4pRPB+5YgjQFhm53xkzU2D4uBbj+VKa0GFw4sJtw0euGJNGCSXQcXTI4yNXDK7KIqLYDkcvs2/4yBUD+C5SSlnjOJ8JMXzkijNM/FAmBMd5eZkAcuV4XEhYETpzxpkANlEsScehqfM4NBPApgKPXzM6YclcyzXjY1MyHjlkahwd8vhglITwfKag8LAwfoY946NPQqYmi9Q4Oh6Xjz5JhPJ0qMaRg70ZH5vwVGeBjuU4OjYPH4wwubUwUaLGkZMJyfhQlaAlwM6w4zjv1OV8qEoSPPlPIzWO82st50NVkhiMhcjSFS7oyPlQlWDOH7tujaNDngBUYUCNHa7H0SFPAKoMHXYbNY5eljwXgKqMkraUPUm163EFoIosOTlmsdMxywXACOB8YSigNo8C6oNXfK4TPIoXwKYMDR/tThz93q4AGGHdSkpuYuo8AigEwAgD6ohOPKLUlZ4pBMAII9o8ohgrcm3WQgCM0G/C2/jG0SGPD0appjMNen/O496CD0ZpRMZEq3GcN14FH31Syt2mlC1NeY5LwcemNMbzyIIOu4tnhrBnHImCj1wphnjYZ3McHe+eD1XYdXQRUgkojo53z8emNEW3LY7UOM4+HzLs2fIMWh4qkcTRIY+PTSlCv6EgwLiCAC1QgJ5m6LhQvily5Zu0QMW5CbE0JKHj5eTY0dWH+OpMnIXi+VBl8MQxpIOW0ITzy1ULFKCbEMuewkKNo2N6+WBkMHmKPczH0SGPDzcGQ/SYTE3sMjVaoOLc4JlpQrU8iUmO3bIZp+zwOvnoYzBETyLC+sjPMdMC9ehGU9FeqMZxfrYF6tENntEiEWIcHfL4YGQw+Z1QkWBCRYLhI3ke50paoDzdRFjeTG5bTG7bM7U54WmBNgLIhU5PTiFQ7gqBtEDxOhZILEIyDKHTMAgUr6M1X+CVLuPokCeAXDEhJR2BaxcyC5SnGzzpiCnDE2sXUgrUo5skwoLDXI3jvDyBenSDyf3Y0POZKSbb8/euQLW6MVRZVKhxdDy9AHJRwoCQy7jKm7VAebrJcDVRPi125dO0QHk6ZkEWhnance5OgfJ0A2HpIqQEDI4OeXz0wbVLR87xyZGzw6wL1KpnIUIt+cQ4HnnEznMHLVC6jtwrsCmRGkfHXPORKQNruEgp04/jvDyBavQMvRhNSK+1H3NFC5SfZwmesNNBA46Ox+VDUYYZAEyzjKNDHh+KMswAaEpF4+iQx4eiLEXWIJ3S4chxywSq0bOUMsVURZlr19MLAFWK1Th0xpymTyv3nmliD9oIIJdJ8WydgJOKt8NHyDVWj0yEY6fFJFqgdD1Do5hSsJ2GBW9lCMAcdubUVNyhXcUdml/JrpF+QS/ClbLX/DJ22IEx1XHELuqh5hetYxiGVzeMo0MaG8/wUDyh6rrExaHV/JJ0DOqwD9o4OqSx0SrGciOaySQ5hNfPNIT8gvMYVyjRduLwKdf8mRvzMBFs4AIztEhCImaHzgXMBiY626ecVfQoZ3UAyWgaJKfeAhuWkAloyDkwxRS56PlRGr8MHR2jkFL5OHJMGb8m3WA+b5HRQUZ2f5DxMuPBr1hHxhkVNEeugmbNL1fPC+rKQnVdmauuS7PL1RHOU4qUU2ekzK5VB4d7kWsKbbQLX9ml6FGKuJJTwVruKljT7Er0iM6mEjo3xzE8wvNJcnF05uNTnGGXqWPjm5BCrtCZXGPXqGPXG0NMIhznhbEL1OmOLqQ/JFQTnriejl2DThdyYaUeRcyxM2JmF6GTwAzrz40aR4dALgLRJVHwhDn5r7nTf2XXnZNAWqEUJTsTWuzSchKIRScZTWnmnFIuFNGlTDFCLDFbXSV+ml1cTgKxIwmFppmLKqHZ5eQkEHt0kAORkwNxwD83hVCzy8vpChdKnCVqHOcfmF1fTgLReyP7Ypz2hV1gTgJx0ZKjjOPD/E4U5Wp2hTldSIPkd8Ly3Inl7BJzEkj8F/J9ilzNFwSwa8zpwg5kvGgSqJ0CJYAnx3dIEW1ueIEXu+icrt+gFoD0/GbqVPV8ewPNLjMn+ejn08lClj+t+3bk+tll5yQfzx3pXA5Hr2Q/u9Cc5JMdItDKXKDFrjSn2yHQKSaSfx67Fjy71JwEIrGdnBfjdF7YteZ0uYJB3jZl+oznK2XXntN9CVg/S5ASaVf8xi4+xzyOpiSrdlY0sUvN8ZDeEPcUR6/Nyi47p2aTVEmJo59sLlDh6ScVKIeuAmXNrjdPcIOmFM6kznCGXV6ejPaIcp65M+fJri5HFw15a+PoEMYFHzyZMkWoxtEhjAs81NWKGljErn6Dml1UjsUdERmNyGk02BXlWEuRUd1VlrgMBrucnOptqQ+ecfXB0+xqcWqYSUfBsfMomF0MTuxQOtbH0YfDotmF4ejkxNR3GUfHg3IhBXsaZdSALHM1IIvYJeHEgaXOKTjOCfsfnxSRVg==', 'recorded_summary': {'top1': 74.2, 'hit3': 86.1, 'hit5': 92.30000000000001, 'mrr': 80.3, 'map': 81.5}}}

def load_recorded_trials(release: str) -> pd.DataFrame:
    record = RECORDED_INPUTS[release]
    raw = zlib.decompress(base64.b64decode(record["data"]))
    if hashlib.sha256(raw).hexdigest() != record["csv_sha256"]:
        raise ValueError(f"Input integrity mismatch: {release}")
    return pd.read_csv(io.BytesIO(raw), keep_default_na=False, dtype={key: str for key in KEYS})

trials = {release: load_recorded_trials(release) for release in RECORDED_INPUTS}
display(trials["before"].head(3))


## 2. Scenario and Trial Validation


In [ ]:
def validate_trials(frame: pd.DataFrame) -> None:
    required = set(KEYS) | {f"{m}_trial{t}" for m in METRICS for t in (1, 2)}
    if not required.issubset(frame.columns):
        raise ValueError(f"Missing columns: {sorted(required - set(frame.columns))}")
    if len(frame) != 500:
        raise ValueError("Expected exactly 500 scenarios")
    for key in ("scenario_id", "source_identifier"):
        if frame[key].eq("").any() or frame[key].duplicated().any():
            raise ValueError(f"Blank or duplicate key: {key}")
    counts = frame["scenario_type"].value_counts().to_dict()
    if counts != {"parent_placement": 300, "reference_lookup": 200}:
        raise ValueError(f"Unexpected scenario composition: {counts}")
    for trial in (1, 2):
        for metric in METRICS:
            values = pd.to_numeric(frame[f"{metric}_trial{trial}"], errors="raise")
            if not np.isfinite(values).all() or not values.between(0, 1).all():
                raise ValueError(f"Invalid values: {metric}, trial {trial}")
            if metric in ("top1", "hit3", "hit5") and not values.isin([0, 1]).all():
                raise ValueError(f"Nonbinary hit values: {metric}, trial {trial}")
        top1, hit3, hit5 = (frame[f"{m}_trial{trial}"] for m in ("top1", "hit3", "hit5"))
        if ((top1 > hit3) | (hit3 > hit5)).any():
            raise ValueError(f"Inconsistent hit ordering in trial {trial}")

for frame in trials.values():
    validate_trials(frame)
pd.testing.assert_frame_equal(trials["before"][list(KEYS)], trials["after"][list(KEYS)])
print("Validated: 500 matched scenarios, 300 parent cases and 200 reference cases.")


## 3. Scenario Averages and Overall Metrics


In [ ]:
def aggregate_trials(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    details = frame[list(KEYS)].copy()
    for metric in METRICS:
        columns = [f"{metric}_trial1", f"{metric}_trial2"]
        details[columns] = frame[columns].astype(float)
        details[metric] = details[columns].mean(axis=1)
    summary = pd.DataFrame({
        "metric": list(METRICS),
        "computed_percent": [details[m].mean() * 100 for m in METRICS],
    })
    return details, summary

rebuilt = {release: aggregate_trials(frame) for release, frame in trials.items()}
comparison = pd.DataFrame({
    "metric": list(METRICS),
    "before_percent": rebuilt["before"][1]["computed_percent"],
    "after_percent": rebuilt["after"][1]["computed_percent"],
})
comparison["difference_percentage_points"] = comparison["after_percent"] - comparison["before_percent"]
for release, (_, summary) in rebuilt.items():
    expected = [RECORDED_INPUTS[release]["recorded_summary"][m] for m in METRICS]
    np.testing.assert_allclose(summary["computed_percent"], expected, atol=ATOL, rtol=0)
display(comparison.round(6))


## 4. Excel Export


In [ ]:
def export_workbook(release: str, details: pd.DataFrame, summary: pd.DataFrame) -> Path:
    destination = OUTPUT / f"HLO_500_{release}_overall_Details.xlsx"
    with pd.ExcelWriter(destination, engine="openpyxl") as writer:
        details.to_excel(writer, sheet_name="scenarios", index=False)
        summary.to_excel(writer, sheet_name="Target", index=False)
        for sheet in writer.book.worksheets:
            sheet.freeze_panes = "A2"
            sheet.auto_filter.ref = sheet.dimensions
            for cell in sheet[1]:
                cell.font = Font(bold=True, color="FFFFFF")
                cell.fill = PatternFill("solid", fgColor="17365D")
                cell.alignment = Alignment(wrap_text=True, vertical="center")
            sheet.row_dimensions[1].height = 32
            for column in sheet.columns:
                letter = column[0].column_letter
                width = max(len(str(cell.value or "")) for cell in column)
                sheet.column_dimensions[letter].width = min(max(width + 2, 14), 55)
            for row in sheet.iter_rows(min_row=2):
                for cell in row:
                    if isinstance(cell.value, float):
                        cell.number_format = "0.000000"
    actual = pd.read_excel(destination, sheet_name="scenarios", keep_default_na=False)
    actual_summary = pd.read_excel(destination, sheet_name="Target")
    pd.testing.assert_frame_equal(actual, details, check_dtype=False, atol=ATOL, rtol=0)
    pd.testing.assert_frame_equal(actual_summary, summary, check_dtype=False, atol=ATOL, rtol=0)
    return destination

output_paths = {release: export_workbook(release, *frames) for release, frames in rebuilt.items()}
comparison.to_csv(OUTPUT / "comparison_summary.csv", index=False)
for release, path in output_paths.items():
    print(f"{release}: {path}")


## 5. Comparison with the Supplied Workbooks


In [ ]:
def locate_evaluation() -> Path | None:
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / "evaluation", parent / "HLO-main" / "evaluation"):
            if (candidate / "compare_ontologies.py").is_file():
                return candidate
    return None

EVALUATION = locate_evaluation()
original_checks = {}
for release, (details, summary) in rebuilt.items():
    original = EVALUATION / "results" / RECORDED_INPUTS[release]["source_name"] if EVALUATION else None
    if original is None or not original.is_file():
        original_checks[release] = "not_run_original_workbook_unavailable"
        continue
    source = pd.read_excel(original, sheet_name=0, keep_default_na=False)
    pd.testing.assert_frame_equal(source[list(details.columns)], details,
                                  check_dtype=False, atol=ATOL, rtol=0)
    original_summary = pd.read_excel(original, sheet_name=1)
    pd.testing.assert_frame_equal(original_summary, summary, check_dtype=False, atol=ATOL, rtol=0)
    original_checks[release] = "all_500_rows_and_5_summary_metrics_match"
print(original_checks)


## 6. Optional Ontology Evaluation


In [ ]:
RUN_ONTOLOGY_EVALUATION = False
ontology_status = "not_run"
if RUN_ONTOLOGY_EVALUATION:
    if EVALUATION is None:
        raise FileNotFoundError("Place the notebook inside the HLO project for ontology evaluation")
    import importlib.util
    spec = importlib.util.spec_from_file_location("hlo_compare", EVALUATION / "compare_ontologies.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    before_summary, after_summary, paired = module.compare(
        EVALUATION / "ontologies/before/HLO_T1.owl",
        EVALUATION / "ontologies/after/HLO_Template_LLM.owl",
        EVALUATION / "data/scenarios.csv",
        OUTPUT / "ontology_checks",
    )
    display(pd.DataFrame(paired))
    ontology_status = "completed_structural_evaluation_only"
else:
    print("Optional ontology evaluation not run.")


## 7. Execution Report


In [ ]:
report = {
    "operation": "rebuild_workbooks_from_recorded_trial_metrics",
    "scenario_count": 500,
    "trial_origin": "not_documented_in_supplied_project",
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "input_integrity": "verified_sha256",
    "scenario_validation": "passed",
    "original_workbook_checks": original_checks,
    "excel_roundtrip": "passed_for_both_workbooks",
    "ontology_evaluation": ontology_status,
    "sources": {key: {k: v for k, v in record.items() if k != "data"}
                for key, record in RECORDED_INPUTS.items()},
    "outputs": {key: {"name": path.name, "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
                for key, path in output_paths.items()},
}
(OUTPUT / "reproduction_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("Workbook reconstruction and verification completed.")
